# Aggregated VisType Scene Graph

**Goal:** Build a static, aggregated scene graph for a chosen visualization type.
Each **node** is a unique synset (e.g. `mark.bar`, `furniture.legend`).
Each **edge** is a canonical relationship triple `(subj_synset → pred_canon → obj_synset)`
that appears in ≥ N images of that VisType.

| Visual element | Meaning |
|---|---|
| Node color | Synset category (mark / property / structure / text / furniture / content / whole) |
| Node size | Number of distinct images the synset appears in |
| Edge color | Dominant sentiment — 🔴 red `[+]` increases complexity, 🔵 blue `[-]` aids comprehension, ⚫ gray = neutral |
| Edge width | Proportional to image count for that triple |
| Edge label | `pred_canon (n=X)` |
| Edge annotation | Top subject attribute → top object attribute (if frequent enough) |

**Input:** `vc_genome_output_full/vistype_profile/oar_relationships_long.csv`  
**Output:** PNG saved to `vc_genome_output_full/vistype_profile/scene_graphs/`


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import graphviz as gv
from collections import Counter, defaultdict
from pathlib import Path

# ── Graphviz binary path (mirrors draw_scene_graphs.py) ──────────────────
os.environ['PATH'] = r'C:\Program Files (x86)\Graphviz\bin;' + os.environ.get('PATH', '')

sys.path.insert(0, 'scripts')
ROOT = Path('.')
OUT  = ROOT / 'vc_genome_output_full' / 'vistype_profile'
GRAPH_OUT = OUT / 'scene_graphs'
GRAPH_OUT.mkdir(parents=True, exist_ok=True)

# ── Visual constants ──────────────────────────────────────────────────────
CATEGORY_COLORS = {
    'mark':      '#4472C4',   # blue
    'property':  '#ED7D31',   # orange
    'structure': '#70AD47',   # green
    'text':      '#FF4444',   # red
    'furniture': '#9B59B6',   # purple
    'content':   '#795548',   # brown
    'whole':     '#90A4AE',   # blue-grey
}
SENTIMENT_COLORS = {
    '[+]': '#E74C3C',   # red  — increases complexity / difficulty
    '[-]': '#2980B9',   # blue — reduces complexity / aids comprehension
    '':    '#AAAAAA',   # grey — neutral or missing
}
print('Libraries loaded.')


Libraries loaded.


In [2]:
rel_df = pd.read_csv(OUT / 'oar_relationships_long.csv')
print(f'Loaded {len(rel_df):,} relationship triples  |  '
      f'{rel_df["imageName"].nunique()} images  |  '
      f'{rel_df["VisType"].nunique()} VisTypes')
print(f'VisTypes: {sorted(rel_df["VisType"].unique())}')


Loaded 916 relationship triples  |  414 images  |  9 VisTypes
VisTypes: ['Area', 'Bar', 'Cont.-ColorPatn', 'Glyph', 'Grid', 'Line', 'Node-link', 'Point', 'Text']


## Build Scene Graph

`build_scene_graph()` — filters to one VisType, aggregates triples by `(subj_synset, pred_canon, obj_synset)`,
and returns a `networkx.MultiDiGraph` plus summary DataFrames.

**Two independent thresholds:**
- `min_node_images` — a synset must appear in ≥ N distinct images (in any role) to become a node. This prunes rare synsets.
- `min_edge_images` — a specific triple `(subj, pred, obj)` must appear in ≥ N distinct images to become an edge. Default = 1 (no edge filtering; all triples between surviving nodes are shown).

> Setting `min_node_images=3, min_edge_images=1` shows **all relationships** between common synsets,
> rather than only the most-repeated specific triples.

**Edge data stored per triple:**
- `n_images` — distinct images where this triple appears
- `sentiment` — dominant `[+]` / `[-]` across those images
- `subj_top_attrs` / `obj_top_attrs` — most salient canonical attributes (appear in ≥ 35 % of instances)


In [3]:
def _top_attrs(series, threshold=0.35, top_n=2):
    """Return the most frequent attrs that each appear in >= threshold fraction of non-empty values."""
    ctr = Counter()
    total = 0
    for val in series.dropna():
        for a in str(val).split(';'):
            a = a.strip()
            if a and a != 'nan':
                ctr[a] += 1
                total += 1
    if not total:
        return ''
    return '; '.join(a for a, c in ctr.most_common(top_n) if c / total >= threshold)


def build_scene_graph(rel_df, vistype, min_node_images=3, min_edge_images=1):
    """
    Build an aggregated scene graph for a given VisType.

    Parameters
    ----------
    rel_df           : DataFrame from oar_relationships_long.csv
    vistype          : str  e.g. 'Bar'
    min_node_images  : int  a synset must appear in this many distinct images
                            (in any role) to be included as a node
    min_edge_images  : int  a triple (subj, pred, obj) must appear in this many
                            distinct images to be included as an edge (default=1
                            means no extra edge filtering beyond node filtering)

    Returns
    -------
    G        : nx.MultiDiGraph  (None if no edges pass threshold)
    node_df  : DataFrame indexed by synset — n_images, category
    edge_df  : DataFrame of filtered triples — counts, sentiment, top attrs
    """
    vt = rel_df[rel_df['VisType'] == vistype].copy()
    if vt.empty:
        raise ValueError(f'No data found for VisType={vistype!r}')

    # ── Node counts (all triples, no threshold yet) ──────────────────────────
    all_synsets = pd.concat([vt['subj_synset'], vt['obj_synset']])
    node_img_counts = (
        pd.concat([
            vt[['subj_synset', 'imageName']].rename(columns={'subj_synset': 'synset'}),
            vt[['obj_synset',  'imageName']].rename(columns={'obj_synset':  'synset'}),
        ])
        .groupby('synset')['imageName'].nunique()
    )
    surviving_nodes = set(node_img_counts[node_img_counts >= min_node_images].index)

    # ── Edge aggregation — only triples whose both nodes survive ─────────────
    edge_rows = []
    for (s_syn, pred, o_syn), grp in vt.groupby(
            ['subj_synset', 'pred_canon', 'obj_synset']):
        if s_syn not in surviving_nodes or o_syn not in surviving_nodes:
            continue
        n_img = grp['imageName'].nunique()
        if n_img < min_edge_images:
            continue
        sents = grp['sentiment'].value_counts()
        dominant_sent = sents.index[0] if (len(sents) and sents.index[0] in SENTIMENT_COLORS) else ''
        edge_rows.append({
            'subj':           s_syn,
            'pred':           pred,
            'obj':            o_syn,
            'n_images':       n_img,
            'sentiment':      dominant_sent,
            'subj_top_attrs': _top_attrs(grp['subj_attrs']),
            'obj_top_attrs':  _top_attrs(grp['obj_attrs']),
        })

    edge_df = pd.DataFrame(edge_rows) if edge_rows else pd.DataFrame(
        columns=['subj', 'pred', 'obj', 'n_images', 'sentiment',
                 'subj_top_attrs', 'obj_top_attrs'])

    if edge_df.empty:
        print(f'No triples pass thresholds for VisType={vistype!r}  '
              f'(min_node={min_node_images}, min_edge={min_edge_images})')
        return None, None, edge_df

    # ── Node DataFrame — restrict to nodes actually used in edges ────────────
    used_nodes = sorted(set(edge_df['subj']) | set(edge_df['obj']))
    node_rows = []
    for syn in used_nodes:
        cat = syn.split('.')[0] if '.' in syn else syn
        node_rows.append({'synset': syn,
                          'n_images': node_img_counts.get(syn, 0),
                          'category': cat})
    node_df = pd.DataFrame(node_rows).set_index('synset')

    # ── Build MultiDiGraph ───────────────────────────────────────────────────
    G = nx.MultiDiGraph()
    for syn, row in node_df.iterrows():
        G.add_node(syn, **row.to_dict())
    for _, row in edge_df.iterrows():
        G.add_edge(row['subj'], row['obj'],
                   pred=row['pred'],
                   n_images=row['n_images'],
                   sentiment=row['sentiment'],
                   subj_top_attrs=row['subj_top_attrs'],
                   obj_top_attrs=row['obj_top_attrs'])

    return G, node_df, edge_df

print('build_scene_graph() defined.')


build_scene_graph() defined.


## Draw Scene Graph

`draw_scene_graph()` — matplotlib static rendering.

- **Layout:** graphviz `dot` (hierarchical) if available, otherwise `kamada_kawai`
- **Parallel edges** between the same node pair get progressively wider arc radii so they don't overlap
- **Edge midpoint labels** are offset along the arc's perpendicular to sit on the curve, not the chord


In [ ]:
def _arc_radii(G):
    """
    Assign a distinct arc radius to every edge in a MultiDiGraph so that
    parallel edges between the same node pair don't overlap.
    Sequence: 0.15, -0.15, 0.30, -0.30, ...
    """
    pair_edges = defaultdict(list)
    for u, v, k in G.edges(keys=True):
        pair_edges[(u, v)].append(k)

    arc_map = {}
    for (u, v), keys in pair_edges.items():
        radii = []
        for i, _ in enumerate(keys):
            sign = 1 if i % 2 == 0 else -1
            mag  = 0.15 * (i // 2 + 1)
            radii.append(sign * mag)
        for k, r in zip(keys, radii):
            arc_map[(u, v, k)] = r
    return arc_map


_CAT_ORDER = ['mark', 'property', 'structure', 'text', 'furniture', 'content', 'whole']


def _compute_layout(G, node_df, layout='multipartite'):
    """
    Compute node positions.
    layout: 'dot_lr'       — graphviz dot, left-to-right (matches per-image scripts)
            'multipartite' — one vertical column per synset category
            'kamada_kawai' — spring / force-directed
    """
    if layout == 'kamada_kawai':
        return nx.kamada_kawai_layout(G)

    if layout == 'multipartite':
        for n in G.nodes():
            cat = node_df.loc[n, 'category']
            G.nodes[n]['subset'] = (
                _CAT_ORDER.index(cat) if cat in _CAT_ORDER else len(_CAT_ORDER)
            )
        return nx.multipartite_layout(G, subset_key='subset', align='vertical')

    # dot_lr — use graphviz package directly (no pygraphviz/pydot needed)
    try:
        nodes = list(G.nodes())
        idx   = {n: f'n{i}' for i, n in enumerate(nodes)}
        rev   = {v: k for k, v in idx.items()}
        dot   = gv.Digraph(graph_attr={'rankdir': 'LR'})
        for n in nodes:
            dot.node(idx[n])
        for u, v in G.edges():
            dot.edge(idx[u], idx[v])
        plain = dot.pipe(format='plain', quiet=True).decode('utf-8')
        pos = {}
        for line in plain.splitlines():
            parts = line.split()
            if parts and parts[0] == 'node' and parts[1] in rev:
                pos[rev[parts[1]]] = (float(parts[2]), float(parts[3]))
        return pos
    except Exception as e:
        print(f'  graphviz layout failed ({e}) — falling back to kamada_kawai')
        return nx.kamada_kawai_layout(G)


def draw_scene_graph(G, node_df, edge_df, vistype,
                     min_images=3, figsize=(22, 14),
                     out_path=None, layout='multipartite'):
    """Render the aggregated scene graph as a static matplotlib figure.

    layout: 'dot_lr' | 'multipartite' | 'kamada_kawai'
    """
    fig, ax = plt.subplots(figsize=figsize)

    # ── Layout ───────────────────────────────────────────────────────────────
    pos = _compute_layout(G, node_df, layout)

    # ── Nodes ────────────────────────────────────────────────────────────────
    node_order  = list(G.nodes())
    node_colors = [CATEGORY_COLORS.get(node_df.loc[n, 'category'], '#CCCCCC')
                   for n in node_order]
    node_sizes  = [120 + node_df.loc[n, 'n_images'] * 55 for n in node_order]

    nx.draw_networkx_nodes(G, pos, nodelist=node_order,
                           node_color=node_colors, node_size=node_sizes,
                           alpha=0.92, ax=ax)
    nx.draw_networkx_labels(G, pos,
        labels={n: f"{n}\n({node_df.loc[n, 'n_images']})" for n in node_order},
        font_size=6.5, font_weight='bold', ax=ax)

    # ── Edges ────────────────────────────────────────────────────────────────
    arc_map = _arc_radii(G)

    for u, v, k, data in G.edges(keys=True, data=True):
        rad   = arc_map.get((u, v, k), 0.15)
        color = SENTIMENT_COLORS.get(data.get('sentiment', ''), '#AAAAAA')
        lw    = 0.7 + data.get('n_images', 1) * 0.28

        nx.draw_networkx_edges(
            G, pos, edgelist=[(u, v)],
            edge_color=color, width=lw, alpha=0.72,
            connectionstyle=f'arc3,rad={rad}',
            arrowsize=13, arrowstyle='-|>',
            min_source_margin=20, min_target_margin=20,
            ax=ax)

        # ── Edge label: offset along arc perpendicular ────────────────────
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        mx, my = (x0 + x1) / 2, (y0 + y1) / 2
        dx, dy = x1 - x0, y1 - y0
        chord_len = np.hypot(dx, dy)
        if chord_len > 0:
            perp = np.array([-dy, dx]) / chord_len
            mx += perp[0] * rad * chord_len * 0.38
            my += perp[1] * rad * chord_len * 0.38

        pred_label = f"{data['pred']} (n={data['n_images']})"
        sa = (data.get('subj_top_attrs') or '').strip()
        oa = (data.get('obj_top_attrs')  or '').strip()
        attr_line = f"\n[{sa or '—'} → {oa or '—'}]" if (sa or oa) else ''

        ax.text(mx, my, pred_label + attr_line,
                fontsize=5, ha='center', va='center', color=color,
                bbox=dict(boxstyle='round,pad=0.18', fc='white',
                          ec='none', alpha=0.82))

    # ── Legend ────────────────────────────────────────────────────────────────
    present_cats = node_df['category'].unique()
    cat_patches = [mpatches.Patch(color=CATEGORY_COLORS.get(c, '#ccc'), label=c)
                   for c in CATEGORY_COLORS if c in present_cats]
    sent_patches = [
        mpatches.Patch(color=SENTIMENT_COLORS['[+]'],  label='[+] increases complexity'),
        mpatches.Patch(color=SENTIMENT_COLORS['[-]'],  label='[-] aids comprehension'),
        mpatches.Patch(color=SENTIMENT_COLORS[''],     label='neutral / no sentiment'),
    ]
    ax.legend(handles=cat_patches + sent_patches,
              loc='lower left', fontsize=7.5, framealpha=0.92,
              title='Node category  /  Edge sentiment', title_fontsize=7.5,
              ncol=2)

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    ax.set_title(
        f'Aggregated OAR Scene Graph  —  VisType: {vistype}  [{layout}]\n'
        f'{n_nodes} synset nodes · {n_edges} relationship edges  '
        f'(threshold ≥ {min_images} images)',
        fontsize=11, pad=10)
    ax.axis('off')
    plt.tight_layout()

    if out_path:
        fig.savefig(out_path, dpi=200, bbox_inches='tight')
        print(f'Saved → {out_path}')
    return fig

print('draw_scene_graph() defined.')


draw_scene_graph() defined.


## Run — Choose VisType and Threshold

Edit `TARGET_VT` and `MIN_IMAGES` then re-run this cell.
Available VisTypes: `Area · Bar · Cont.-ColorPatn · Glyph · Grid · Line · Node-link · Point · Text`


In [35]:
# ── Parameters ───────────────────────────────────────────────────────────────
TARGET_VT       = 'Grid'   # change to any VisType
MIN_NODE_IMAGES = 2   # synset must appear in ≥ N images (any role) to be a node
MIN_EDGE_IMAGES = 1   # triple must appear in ≥ N images to be an edge (1 = off)

# ── Build ─────────────────────────────────────────────────────────────────────
G, node_df, edge_df = build_scene_graph(rel_df, TARGET_VT,
                                         min_node_images=MIN_NODE_IMAGES,
                                         min_edge_images=MIN_EDGE_IMAGES)

if G is not None:
    print(f'Nodes : {G.number_of_nodes()}')
    print(f'Edges : {G.number_of_edges()}')
    print(f'\nNode summary (sorted by image count):')
    display(node_df.sort_values('n_images', ascending=False))
    print(f'\nEdge summary (sorted by image count):')
    display(edge_df.sort_values('n_images', ascending=False))


Nodes : 17
Edges : 97

Node summary (sorted by image count):


,n_images,category
synset,,
whole.visualization,35,whole
property.color,29,property
content.data,16,content
text.label,16,text
mark.shape,12,mark
mark.element,6,mark
mark.point,6,mark
structure.layout,5,structure
furniture.axes,5,furniture



Edge summary (sorted by image count):


,subj,pred,obj,n_images,sentiment,subj_top_attrs,obj_top_attrs
56,property.color,increases_complexity,whole.visualization,4,[+],,high_information_volume; confusing_and_complex
57,property.color,increases_effort,whole.visualization,2,[+],multiple_colors_present; meaning_unclear_to_vi...,
94,whole.visualization,requires_expertise,content.data,2,[+],,
6,content.data,increases_effort,whole.visualization,2,[+],,requires_extended_processing_time
0,content.data,adds_nuance_to,whole.visualization,1,[+],nuanced_visual_features,not_easily_decipherable
...,...,...,...,...,...,...,...
90,whole.visualization,has_low_clutter_due_to,furniture.legend,1,[-],low_information_density,few_legend_items
92,whole.visualization,increases_cognitive_load_via,content.data,1,[+],high_image_count,high_information_volume
93,whole.visualization,lacks_interpretable,content.data,1,[+],,no_data_present; nothing_visible
95,whole.visualization,requires_specialized_knowledge,whole.visualization,1,[+],technical_presentation_style,technical_presentation_style


In [ ]:
if G is not None:
    slug = TARGET_VT.lower().replace('-', '_').replace('.', '').replace(' ', '_')
    out_path = GRAPH_OUT / f'scene_graph_{slug}_node{MIN_NODE_IMAGES}_edge{MIN_EDGE_IMAGES}.png'
    fig = draw_scene_graph(G, node_df, edge_df, TARGET_VT,
                           min_images=MIN_NODE_IMAGES, out_path=out_path,
                           layout='multipartite')
    plt.close(fig)


Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_point_node2_edge2.png


## Layout Comparison

Render the same graph with all three layout strategies and save one PNG each.


In [29]:
if G is not None:
    slug = TARGET_VT.lower().replace('-', '_').replace('.', '').replace(' ', '_')
    for layout in ('dot_lr', 'multipartite', 'kamada_kawai'):
        out_path = GRAPH_OUT / f'scene_graph_{slug}_node{MIN_NODE_IMAGES}_edge{MIN_EDGE_IMAGES}_{layout}.png'
        fig = draw_scene_graph(G, node_df, edge_df, TARGET_VT,
                               min_images=MIN_NODE_IMAGES, out_path=out_path,
                               layout=layout)
        plt.close(fig)


Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_bar_node2_edge1_dot_lr.png
Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_bar_node2_edge1_multipartite.png
Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_bar_node2_edge1_kamada_kawai.png


## Batch — All VisTypes

Run this cell to generate one PNG per VisType and print a quick coverage summary.


In [36]:
BATCH_MIN_NODE = 2   # synset must appear in ≥ N images to be a node
BATCH_MIN_EDGE = 1   # triple must appear in ≥ N images to be an edge (1 = off)

summary_rows = []
for vt in sorted(rel_df['VisType'].unique()):
    try:
        G_b, node_df_b, edge_df_b = build_scene_graph(rel_df, vt,
                                                        min_node_images=BATCH_MIN_NODE,
                                                        min_edge_images=BATCH_MIN_EDGE)
    except ValueError as e:
        print(f'  {vt}: {e}')
        continue
    if G_b is None:
        summary_rows.append({'VisType': vt, 'nodes': 0, 'edges': 0, 'status': 'no edges'})
        continue

    slug = vt.lower().replace('-', '_').replace('.', '').replace(' ', '_')
    out_path = GRAPH_OUT / f'scene_graph_{slug}_node{BATCH_MIN_NODE}_edge{BATCH_MIN_EDGE}.png'
    draw_scene_graph(G_b, node_df_b, edge_df_b, vt,
                     min_images=BATCH_MIN_NODE, out_path=out_path)
    plt.close('all')
    summary_rows.append({
        'VisType': vt,
        'nodes':   G_b.number_of_nodes(),
        'edges':   G_b.number_of_edges(),
        'status':  'saved',
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_area_node2_edge1.png
Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_bar_node2_edge1.png
Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_cont_colorpatn_node2_edge1.png
Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_glyph_node2_edge1.png
Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_grid_node2_edge1.png
Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_line_node2_edge1.png
Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_node_link_node2_edge1.png
Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_point_node2_edge1.png
Saved → vc_genome_output_full\vistype_profile\scene_graphs\scene_graph_text_node2_edge1.png


,VisType,nodes,edges,status
0,Area,14,100,saved
1,Bar,12,52,saved
2,Cont.-ColorPatn,15,66,saved
3,Glyph,17,81,saved
4,Grid,17,97,saved
5,Line,16,75,saved
6,Node-link,16,108,saved
7,Point,17,110,saved
8,Text,11,71,saved


In [47]:
pred_counts = rel_df.groupby('pred_canon')['imageName'].nunique().sort_values(ascending=False)
pred_counts.head(105)

pred_canon
increases_effort                  69
increases_clutter_in              40
increases_complexity              34
contributes_to                    32
contributes_to_clutter_in         21
                                  ..
contributes_to_clutter_of          2
visually_dominates                 1
requires_reference_to              1
requires_reading_to_understand     1
requires_pattern_finding_in        1
Name: imageName, Length: 105, dtype: int64